# Pre-Arrears Strategy — Component: Bureau Riskier Qualifiers (SAS Step 4)

**Source**: `Pre-Arrears.txt`, lines 318–850 (unwrapped `proc sql` pass-through,
no enclosing macro)

Fourth component of the 10-step strategy, converted independently. This
is the largest single component in the program: it scores each account's
**two** applicants against 17 bureau risk criteria, takes the worse
(maximum) of the two per criterion, and sums them into a single
`BureauRiskScore`.


## 1. What the SAS component does

### Structure

| CTE | Purpose |
|---|---|
| `CTE_PC1` | Postcode-level Geo-Delphi risk reference, built from **the whole Experian extract for the period** (not just the mortgage portfolio) — one row per normalised postcode, keeping the best-ranked (lowest index, then lowest score) record per postcode. |
| `CTE_PC2` | Same idea one level coarser (postcode district instead of full postcode), derived from `CTE_PC1`. |
| `CTE_C1` | Per-account bureau data + 17 risk-criterion scores for **applicant 1** (the bureau record ranked first by `CustomerID` for that account/period/lender), joined to the qualifying mortgage population (same population rule as Steps 1–2: SBS/AHL/NYM, not pool `DRC00001`, Primary/Residential, Live, pre-arrears, not a deed product, balance > 1000, existed before the prior month). Also computes the account's Geo-Delphi index/score (`GD_Index`/`GD_Score`), using the account's **own** bureau record directly for **Resi**, or falling back through `CTE_PC1` → `CTE_PC2` by postcode for **BTL**. |
| `CTE_C2` | Same 17 scores for **applicant 2** (bureau record ranked second), same population filter, no Geo-Delphi (computed once, in C1). |
| `CTE_Merge` | Left-joins C1 to C2 by account; every score/attribute is coalesced `NULL → 0` (or `''` for names). Adds `ExpData` (1 if applicant 1 has bureau data) and `Cust2Data` (1 if **both** applicants have bureau data). |
| `CTE_App` | For each of the 17 criteria (+ CII, AAM), takes `MAX(C1, C2)` as the "application-level" score — the worse of the two applicants. |
| `CTE_Final` | Sums the 17 application-level criteria into `BureauRiskScore`; keeps `AccountNumber`, `Lending`, `ExpData`, `Cust2Data`, `GD_Index`, `GD_BTLProxy2`, `BureauRiskScore`, and every individual `_App` score. |

### The 17 scoring criteria (per applicant)

Each is a `CASE WHEN` returning `0`–`3` based on bureau attribute
thresholds: **CLU** (credit-limit utilisation), **BDS** (behavioural
data sharing), **10 Vertical Market scores** (Bank Cards, Credit Sales
Agreements, Retail Storecards, Unsecured Loans, Vehicle Finance/HP,
Secured Loans, Mortgages, Current Accounts, Utilities, Home Shopping),
**HC** (Home Credit/Payday), **PDL** (Payday Loans), **BankIVAPIR**
(bankruptcy/IVA/CCJ/public-record), **NPD_EB** (never-paid
defaults/early bads), **CIFAS** (fraud marker).

The 10 Vertical Market scores use only **two distinct patterns**,
differing only in which raw columns they read:

- **Revolving pattern** (Bank Cards, Retail Storecards, Current
  Accounts, Home Shopping): 4 inputs — worst status L6M, current worst
  status, total CLU%, months since last default.
- **Instalment pattern** (Credit Sales, Unsecured Loans, Vehicle
  Finance/HP, Secured Loans, Mortgages, Utilities): 3 inputs — worst
  status L6M, current worst status, months since last default (no CLU
  gate).

The SAS/T-SQL source repeats each pattern's `CASE WHEN` **10 times**
(once per vertical market) **and again for applicant 2** — 20
near-identical copies of 2 underlying rules. That duplication is
collapsed into two generic, independently-tested functions in the
Python conversion (see §3) — this changes **how** the calculation is
expressed, not what it computes; every threshold is preserved exactly
and tested against the original per-market wording.

### ⚠️ The output is not filtered by risk score — read this before using this component

The final `SELECT` has its threshold `WHERE` clause **commented out**:

```sql
SELECT f.*
FROM CTE_Final f
/*WHERE (f.Lending='Resi' AND f.BureauRiskScore>=&GBRS)
        OR (f.Lending='BTL' AND (f.BureauRiskScore>=&GBRS OR (f.BureauRiskScore>=&SBRS AND f.GD_Index>=&BGD)))*/
ORDER BY f.AccountNumber
```

`&GBRS`, `&SBRS`, `&BGD` are **not defined anywhere in the supplied
excerpt** — they only appear inside this disabled comment. As the
component actually runs today, it returns **every** account in the
population with its computed `BureauRiskScore`, with **no risk-based
cut applied at this step**. The Python conversion preserves this
exactly: no threshold filter is applied, and this is asserted in code
(§4) so a future accidental re-enable of filtering would fail loudly
rather than silently changing selection. **TODO: VERIFY** whether this
threshold is applied later in the strategy (Step 6, "merge/cut-offs",
per the component list) — if so, `GBRS`/`SBRS`/`BGD` need to be sourced
from wherever that step defines them, not reintroduced here.


## 2. Semantic risks

1. **Threshold filter disabled, not removed** (see box above) — the
   single highest-impact finding in this component. Preserved as-is;
   flagged, not fixed.
2. **Postcode format mismatch risk.** `CTE_C1` joins the account's raw
   `dms.PostCode` (property postcode, unmodified) directly against
   `p1.PC1`, which **is** normalised (spaces stripped then re-inserted
   at a fixed position by string length). The join only succeeds if
   `dms.PostCode` already happens to be stored in that exact
   "OUTWARD INWARD" single-space format. If the property postcode is
   ever stored without a space, with extra whitespace, or
   lower-cased differently from the bureau extract, the match silently
   fails and the account falls through to the `PC2`/`0` path instead of
   raising an error. **TODO: VERIFY the stored format of
   `dimMortgageSecurity.PostCode`** against the Experian normalisation
   logic — this could be silently under-serving Geo-Delphi proxies for
   BTL accounts today, independent of this conversion.
3. **`CTE_PC1`'s postcode reference is built from the whole Experian
   extract for the period, not the mortgage portfolio.** It is a
   market-wide postcode risk lookup, keyed only by postcode, entirely
   independent of which accounts are in this strategy. This must not
   be confused with a per-account calculation — it is computed once and
   shared.
4. **Two different "CustRank" concepts share the same ranking logic but
   different purposes.** The `CustRank` inside `CTE_PC1`'s inner query
   and the `CustRank` used to select applicant 1/2 in `CTE_C1`/`CTE_C2`
   are structurally identical (`RANK() OVER (PARTITION BY Period,
   LenderCode, PrimaryAccount ORDER BY CustomerID ASC)`) but serve
   different roles — one selects a representative bureau record for the
   postcode reference table, the other selects "applicant 1" vs
   "applicant 2" for the actual account-level scoring. `RANK()` (not
   `ROW_NUMBER()`) means **ties in `CustomerID` would produce more than
   one row at the same rank**, fanning out downstream joins.
   **TODO: VERIFY `CustomerID` is unique per (Period, LenderCode,
   PrimaryAccount)** in the source Experian extract — if not, this
   fan-out is a pre-existing behaviour to be aware of, not something
   this conversion introduces or fixes.
5. **NULL / missing bureau data.** Every raw Experian attribute
   participates in `>=`/`>`/`IN` comparisons; under SQL's three-valued
   logic, a `NULL` operand makes the comparison `UNKNOWN`, which behaves
   like `FALSE` for `CASE WHEN` purposes — it falls through to the next
   branch or the `ELSE 0`. Pandas `NaN` comparisons behave the same way
   (evaluate to `False`), so this needs no special-casing, but it is
   validated with explicit tests below rather than assumed.
6. **Coalesce-to-zero at the merge stage is deliberate**, not an
   accident: an applicant with no bureau match contributes `0` to every
   criterion (a "no adverse data found" outcome), not a missing/null
   value that could be excluded from the sum.
7. **`MAX(C1, C2)` uses `>`, not `>=`.** `CASE WHEN c2.X > c1.X THEN c2.X
   ELSE c1.X END` — when scores are equal, the `ELSE` branch (C1)
   applies. Functionally identical to a true max when both are numeric
   and neither is NULL (already coalesced to 0 upstream), but the
   Python replica uses an explicit `max()` — mathematically identical
   result, verified by test, not assumed identical by construction.
8. **Sort order.** Final result ordered by `AccountNumber` — preserved
   explicitly in the Python version.
9. **Duplicate handling / `first.`/`last.`.** No DATA-step BY-group
   logic in this component; the only ranking is the `RANK()` calls
   discussed in risk #4.
10. **Retained values.** Not applicable — pure SQL pass-through, no
    DATA step accumulation.


In [ ]:
"""
bureau_riskier_qualifiers.py
Production module: Pre-Arrears Strategy — Step 4: Bureau Riskier Qualifiers.

Converted from Pre-Arrears.txt lines 318-850 (unnamed proc sql pass-through).

Design: raw bureau attribute extraction stays as SQL Server pass-through
(it depends on warehouse tables not available locally); all 17 scoring
rules are re-expressed as small, independently-tested Python functions,
eliminating the ~20-fold duplication of the two Vertical Market CASE WHEN
patterns in the SAS source (see risk log / section 1 note on repeated
calculations). No threshold/selection logic is applied here -- see the
"disabled WHERE clause" note in the explanation above.
"""

import logging
import re
import sys
from dataclasses import dataclass, field
from datetime import date
from typing import List, Optional

import pandas as pd

logger = logging.getLogger("pre_arrears.bureau_riskier_qualifiers")
if not logger.handlers:
    _handler = logging.StreamHandler(sys.stdout)
    _handler.setFormatter(
        logging.Formatter("%(asctime)s | %(levelname)-8s | %(name)s | %(message)s")
    )
    logger.addHandler(_handler)
    logger.setLevel(logging.INFO)


class BureauQualifierError(Exception):
    """Raised when Step 4 extraction/scoring cannot be trusted to proceed."""


In [ ]:
@dataclass(frozen=True)
class BureauQualifierConfig:
    rundate: date  # must be the 1st day of a calendar month

    db_server: str = "SBSPSQLVS101\\MISPSQL01"
    db_database: str = "P_DW"
    db_driver: str = "ODBC Driver 17 for SQL Server"
    connect_timeout_seconds: int = 30
    max_retries: int = 3
    retry_backoff_seconds: float = 5.0

    company_scope: List[str] = field(default_factory=lambda: [
        "Skipton Building Society", "Amber Homeloans Ltd", "North Yorkshire Mortgages",
    ])
    excluded_account_pool: str = "DRC00001"
    excluded_product_codes: List[str] = field(default_factory=lambda: [
        "MIN01", "HOM10", "HM120", "HM121", "HM122", "HM123", "HM124", "HM125",
    ])
    capital_balance_threshold: float = 1000.0

    # See "threshold filter disabled" note: these exist only so a caller
    # can inspect what the SAS comment referenced. They are NEVER applied
    # by this module -- see `apply_disabled_threshold_filter` below, which
    # exists solely as a documented no-op guarded by an explicit flag.
    gbrs_general_bureau_risk_score: Optional[int] = None
    sbrs_secondary_bureau_risk_score: Optional[int] = None
    bgd_btl_geo_delphi_threshold: Optional[int] = None

    def __post_init__(self):
        if self.rundate.day != 1:
            raise ValueError(f"rundate must be the first day of a month; got {self.rundate}")


In [ ]:
def get_reporting_period(rundate: date) -> int:
    """Equivalent of SAS: intnx(month,&rundate,0) formatted yymmn6. -> int YYYYMM (the &brss macro var)."""
    return rundate.year * 100 + rundate.month


### 3. Postcode normalisation (Geo-Delphi reference, `CTE_PC1` / `CTE_PC2`)

In [ ]:
def normalise_postcode_pc1(raw_postcode: Optional[str]) -> Optional[str]:
    """
    Equivalent of the PC1 CASE expression in CTE_PC1 (lines 332-336):
    strips all spaces, then re-inserts a single space at a fixed position
    based on the *cleaned* (space-stripped) length. Lengths outside
    4-7 are returned unchanged (the SAS/T-SQL ELSE branch).
    """
    if raw_postcode is None:
        return None
    cleaned = raw_postcode.replace(" ", "")
    n = len(cleaned)
    if n == 7:
        return cleaned[:4] + " " + cleaned[4:]
    if n == 6:
        return cleaned[:3] + " " + cleaned[3:]
    if n == 5:
        return cleaned[:2] + " " + cleaned[2:]
    if n == 4:
        return cleaned[:1] + " " + cleaned[1:]
    return raw_postcode  # ELSE: unchanged, including the original spacing


def postcode_pc1_to_pc2(pc1: Optional[str]) -> Optional[str]:
    """
    Equivalent of the PC2 CASE expression in CTE_PC2 (lines 355-359):
    drops the PC1 value's last character for PC1 lengths 5-8 (i.e. the
    space-inserted forms of cleaned lengths 4-7); unchanged otherwise.
    """
    if pc1 is None:
        return None
    n = len(pc1)
    if n in (5, 6, 7, 8):
        return pc1[:-1]
    return pc1


def is_valid_pc1_candidate(pc1: Optional[str]) -> bool:
    """
    Equivalent of CTE_PC1's outer filter (lines 345-346):
    PC1 IS NOT NULL AND LEFT(PC1,1) NOT IN ('0'..'9')
    -- excludes postcode-shaped values that start with a digit (not a
    valid UK postcode outward-code first character).
    """
    if not pc1:
        return False
    return not pc1[0].isdigit()


### 4. Bureau scoring rules (one function per criterion, shared by both applicants)

In [ ]:
def _in_range_inclusive(value, lo, hi) -> bool:
    """NULL-safe range check matching SQL's `value >= lo AND value <= hi` under 3-valued logic."""
    return value is not None and not pd.isna(value) and lo <= value <= hi


def _gt(value, threshold) -> bool:
    return value is not None and not pd.isna(value) and value > threshold


def _ge(value, threshold) -> bool:
    return value is not None and not pd.isna(value) and value >= threshold


def _eq(value, target) -> bool:
    return value is not None and not pd.isna(value) and value == target


def _in_set(value, allowed) -> bool:
    return value is not None and not pd.isna(value) and value in allowed


def score_clu(sp_f3_35, sp_f1_30, sp_f3_36) -> int:
    """Credit Limit Utilisation (lines 401-404)."""
    if _in_range_inclusive(sp_f3_35, 90, 9998) and _gt(sp_f1_30, 2) and _in_range_inclusive(sp_f3_36, 90, 9998):
        return 3
    if _in_range_inclusive(sp_f3_35, 90, 9998) and _gt(sp_f1_30, 2):
        return 2
    if _in_range_inclusive(sp_f3_35, 85, 9998) and _gt(sp_f1_30, 1):
        return 1
    return 0


def score_bds(clu_npr_l1m, ptbr_l6m_npr_l6m, ptbr_l3m_npr_l3m, no_ca_l3m, no_mlv_ca_l3m) -> int:
    """Behavioural Data Sharing (lines 412-415)."""
    clu_gate = _in_range_inclusive(clu_npr_l1m, 90, 9998)
    if clu_gate and _in_range_inclusive(ptbr_l6m_npr_l6m, 0, 5) and (_ge(no_ca_l3m, 3) or _gt(no_mlv_ca_l3m, 0)):
        return 3
    if clu_gate and _in_range_inclusive(ptbr_l6m_npr_l6m, 0, 5):
        return 2
    if clu_gate and _in_range_inclusive(ptbr_l3m_npr_l3m, 0, 5):
        return 1
    return 0


_WORST_STATUS_2_TO_3 = {"4", "5", "6", "7"}
_WORST_STATUS_0_TO_3 = {"0", "1", "2", "3"}


def score_vm_revolving(status_l6m, status_current, clu_pct, months_since_default) -> int:
    """
    Vertical Market score, "revolving" pattern -- used identically for
    Bank Cards, Retail Storecards, Current Accounts, and Home Shopping
    (lines 422-425, 441-444, 487-490, 506-509 -- verified byte-identical
    other than column names).
    """
    if _in_range_inclusive(months_since_default, 1, 12):
        return 3
    if (months_since_default is not None and not pd.isna(months_since_default) and 12 < months_since_default <= 24) \
            or _in_set(status_current, _WORST_STATUS_2_TO_3):
        return 2
    if _in_set(status_current, _WORST_STATUS_0_TO_3) and _in_set(status_l6m, _WORST_STATUS_2_TO_3) and _ge(clu_pct, 75):
        return 1
    return 0


def score_vm_instalment(status_l6m, status_current, months_since_default) -> int:
    """
    Vertical Market score, "instalment" pattern -- used identically for
    Credit Sales Agreements, Unsecured Loans, Vehicle Finance/HP, Secured
    Loans, Mortgages, and Utilities (lines 431-434, 450-453, 459-462,
    468-471, 477-480, 496-499 -- verified byte-identical other than
    column names).
    """
    if _in_range_inclusive(months_since_default, 1, 12):
        return 3
    if (months_since_default is not None and not pd.isna(months_since_default) and 12 < months_since_default <= 24) \
            or _in_set(status_current, _WORST_STATUS_2_TO_3):
        return 2
    if _in_set(status_current, _WORST_STATUS_0_TO_3) and _in_set(status_l6m, _WORST_STATUS_2_TO_3):
        return 1
    return 0


# Maps each Vertical Market category to (pattern, raw column suffix set),
# purely for documentation / query-building use -- NOT used to alter scoring.
VM_REVOLVING_MARKETS = ["BankCards", "RetCards", "CA", "HS"]     # VM01, VM03, VM08, VM10
VM_INSTALMENT_MARKETS = ["CSA", "UL", "VF_HP", "SL", "MG", "UT"]  # VM02, VM04, VM05, VM06, VM07, VM09


def score_hc(hc_a_08, hc_c_05, hc_e_02) -> int:
    """Home Credit & PayDay Loans, HC family (lines 515-518)."""
    if _in_range_inclusive(hc_c_05, 1, 12):
        return 3
    if (hc_c_05 is not None and not pd.isna(hc_c_05) and 12 < hc_c_05 <= 24) or _gt(hc_a_08, 0):
        return 2
    if _eq(hc_a_08, 0) and _gt(hc_e_02, 0):
        return 1
    return 0


def score_pdl(pdl_e_01, pdl_c_01, pdl_f_01) -> int:
    """PayDay Loans, PDL family (lines 523-526)."""
    if _gt(pdl_e_01, 0):
        return 3
    if _eq(pdl_e_01, 0) and _gt(pdl_c_01, 0):
        return 2
    if _eq(pdl_e_01, 0) and _eq(pdl_c_01, 0) and _gt(pdl_f_01, 0):
        return 1
    return 0


def score_bank_iva_pir(main_bankrupt, main_iva, main_ccj, ea1_d_03) -> int:
    """Bankruptcy / IVA / CCJ / Public Information Record (lines 533-536)."""
    is_bankrupt_or_iva = (main_bankrupt == "B") or (main_iva == "V")
    if is_bankrupt_or_iva:
        return 3
    not_bankrupt_or_iva = (main_bankrupt != "B") and (main_iva != "V")
    if not_bankrupt_or_iva and _ge(main_ccj, 500) and _in_range_inclusive(ea1_d_03, 1, 12):
        return 2
    if not_bankrupt_or_iva and _ge(main_ccj, 500) and _gt(ea1_d_03, 12):
        return 1
    return 0


def score_npd_eb(npd_bal_npd_l12m, npd_bal_ebad_l12m, npd_bal_npd_l18m, npd_bal_ebad_l18m) -> int:
    """Never-Paid Defaults & Early Bads (lines 548-551)."""
    if _ge(npd_bal_npd_l12m, 500):
        return 3
    npd_l12m_below_500 = npd_bal_npd_l12m is not None and not pd.isna(npd_bal_npd_l12m) and npd_bal_npd_l12m < 500
    if npd_l12m_below_500 and _ge(npd_bal_ebad_l12m, 500):
        return 2
    if _eq(npd_bal_npd_l12m, 0) and _eq(npd_bal_ebad_l12m, 0) and (_ge(npd_bal_npd_l18m, 500) or _ge(npd_bal_ebad_l18m, 500)):
        return 1
    return 0


_CIFAS_CLEAN_CODES = {"N", "T", "00", "99"}


def score_cifas(sp_c_24) -> int:
    """CIFAS Fraud Category (line 554). NULL SP_C_24 is NOT IN the clean list -> scores 3, per SQL NOT IN semantics
    being NULL-safe here only because the comparison set doesn't include NULL; SQL `NULL NOT IN (...)` is actually
    UNKNOWN (not TRUE), so a NULL SP_C_24 does NOT score 3 in SQL Server -- see risk note below."""
    if sp_c_24 is None or pd.isna(sp_c_24):
        # SQL: `NULL NOT IN ('N','T','00','99')` evaluates to UNKNOWN, which the
        # CASE WHEN treats as not-satisfied -> falls to ELSE 0. Reproduced exactly.
        return 0
    return 3 if sp_c_24 not in _CIFAS_CLEAN_CODES else 0


### 5. Per-applicant aggregation, App-level max, and `BureauRiskScore`

In [ ]:
# Order matters only for readability; BureauRiskScore is a straight sum.
SCORE_FIELD_NAMES = [
    "CLU_Score", "BDS_Score",
    "VM_BankCards_Score", "VM_CSA_Score", "VM_RetCards_Score", "VM_UL_Score",
    "VM_VF_HP_Score", "VM_SL_Score", "VM_MG_Score", "VM_CA_Score",
    "VM_UT_Score", "VM_HS_Score",
    "HC_Score", "PDL_Score", "BankIVAPIR_Score", "NPD_EB_Score", "CIFAS_Score",
]


def score_applicant(raw: dict) -> dict:
    """
    Computes all 17 criteria for one applicant's raw bureau attributes.
    `raw` keys match the Experian column names used in the SQL source
    (see `build_applicant_bureau_query` below for the exact column list).
    Missing keys are treated as NULL (consistent with a LEFT JOIN miss).
    """
    g = raw.get  # shorthand; g(key) returns None for missing keys, matching a bureau non-match
    return {
        "CLU_Score": score_clu(g("SP_F3_35"), g("SP_F1_30"), g("SP_F3_36")),
        "BDS_Score": score_bds(g("CLU_NPR_L1M"), g("PTBR_L6M_NPR_L6M"), g("PTBR_L3M_NPR_L3M"), g("NO_CA_L3M"), g("NO_MLV_CA_L3M")),
        "VM_BankCards_Score": score_vm_revolving(g("VM01_SP_VM2_04"), g("VM01_SP_VM2_05"), g("VM01_SP_VM2_23"), g("VM01_SP_VM2_29")),
        "VM_CSA_Score": score_vm_instalment(g("VM02_SP_VM1_04"), g("VM02_SP_VM1_05"), g("VM02_SP_VM1_21")),
        "VM_RetCards_Score": score_vm_revolving(g("VM03_SP_VM2_04"), g("VM03_SP_VM2_05"), g("VM03_SP_VM2_23"), g("VM03_SP_VM2_29")),
        "VM_UL_Score": score_vm_instalment(g("VM04_SP_VM1_04"), g("VM04_SP_VM1_05"), g("VM04_SP_VM1_21")),
        "VM_VF_HP_Score": score_vm_instalment(g("VM05_SP_VM1_04"), g("VM05_SP_VM1_05"), g("VM05_SP_VM1_21")),
        "VM_SL_Score": score_vm_instalment(g("VM06_SP_VM1_04"), g("VM06_SP_VM1_05"), g("VM06_SP_VM1_21")),
        "VM_MG_Score": score_vm_instalment(g("VM07_SP_VM1_04"), g("VM07_SP_VM1_05"), g("VM07_SP_VM1_21")),
        "VM_CA_Score": score_vm_revolving(g("VM08_SP_VM2_04"), g("VM08_SP_VM2_05"), g("VM08_SP_VM2_23"), g("VM08_SP_VM2_29")),
        "VM_UT_Score": score_vm_instalment(g("VM09_SP_VM1_04"), g("VM09_SP_VM1_05"), g("VM09_SP_VM1_21")),
        "VM_HS_Score": score_vm_revolving(g("VM10_SP_VM2_04"), g("VM10_SP_VM2_05"), g("VM10_SP_VM2_23"), g("VM10_SP_VM2_29")),
        "HC_Score": score_hc(g("HC_A_08"), g("HC_C_05"), g("HC_E_02")),
        "PDL_Score": score_pdl(g("PDL_E_01"), g("PDL_C_01"), g("PDL_F_01")),
        "BankIVAPIR_Score": score_bank_iva_pir(g("MAIN_BANKRUPT"), g("MAIN_IVA"), g("MAIN_CCJ"), g("EA1_D_03")),
        "NPD_EB_Score": score_npd_eb(g("NPD_BAL_NPD_SP_L12M"), g("NPD_BAL_EBAD_SP_L12M"), g("NPD_BAL_NPD_SP_L18M"), g("NPD_BAL_EBAD_SP_L18M")),
        "CIFAS_Score": score_cifas(g("SP_C_24")),
        "CII": g("SP_CONSUMER_INDEBT") or 0,
        "AAM": g("E5_S_05_1") or 0,
        "has_bureau_match": bool(raw.get("_has_match", False)),
    }


def build_customer_name(raw: dict) -> str:
    """Equivalent of: PORTFOLIO_TITLE + ' ' + PORTFOLIO_GIVEN_NM + ' ' + PORTFOLIO_FAM_NM."""
    parts = [raw.get("PORTFOLIO_TITLE") or "", raw.get("PORTFOLIO_GIVEN_NM") or "", raw.get("PORTFOLIO_FAM_NM") or ""]
    return " ".join(parts)


def combine_applicants(applicant1: dict, applicant2: dict) -> dict:
    """
    Equivalent of CTE_Merge -> CTE_App -> CTE_Final's per-criterion App-level
    MAX(C1, C2) and the final BureauRiskScore sum, for a single account.
    Both inputs are already coalesced to 0/'' for missing fields (see
    `score_applicant`, which defaults every numeric criterion to 0 when
    the bureau record is missing).
    """
    exp_data = 1 if applicant1.get("has_bureau_match") else 0
    cust2_data = 1 if applicant1.get("has_bureau_match") and applicant2.get("has_bureau_match") else 0

    app_scores = {name: max(applicant1[name], applicant2[name]) for name in SCORE_FIELD_NAMES}
    cii_app = max(applicant1["CII"], applicant2["CII"])
    aam_app = max(applicant1["AAM"], applicant2["AAM"])

    bureau_risk_score = sum(app_scores.values())

    return {
        "ExpData": exp_data,
        "Cust2Data": cust2_data,
        "CII_App": cii_app,
        "AAM_App": aam_app,
        **{f"{name}_App": value for name, value in app_scores.items()},
        "BureauRiskScore": bureau_risk_score,
    }


### 6. Geo-Delphi index/score resolution (Resi direct, BTL cascading proxy)

In [ ]:
def resolve_geo_delphi(
    lending: str,
    own_bureau_gd_index: Optional[float],
    own_bureau_gd_score: Optional[float],
    pc1_lookup: dict,
    pc2_lookup: dict,
    property_postcode: Optional[str],
) -> dict:
    """
    Equivalent of CTE_C1's GD_Index/GD_Score/GD_BTLProxy2 CASE expressions
    (lines 380-391).

    - Resi: use the account's own bureau record's Geo-Delphi fields
      directly; 0 if absent.
    - BTL: try the property postcode against the PC1 reference table
      first; if absent, fall back to the PC2 (district-level) table;
      0 if neither matches. `GD_BTLProxy2=1` flags the PC2 fallback path
      specifically (matches the source; PC1-level BTL matches are NOT
      flagged as a "proxy" -- only the coarser PC2 fallback is).
    """
    if lending != "BTL":
        gd_index = own_bureau_gd_index if own_bureau_gd_index is not None else 0
        gd_score = own_bureau_gd_score if own_bureau_gd_score is not None else 0
        return {"GD_Index": gd_index, "GD_Score": gd_score, "GD_BTLProxy2": 0}

    pc1_match = pc1_lookup.get(property_postcode)
    if pc1_match is not None:
        return {"GD_Index": pc1_match["GD_Index1"], "GD_Score": pc1_match["GD_Score1"], "GD_BTLProxy2": 0}

    pc2_key = postcode_pc1_to_pc2(property_postcode)
    pc2_match = pc2_lookup.get(pc2_key)
    if pc2_match is not None:
        return {"GD_Index": pc2_match["GD_Index2"], "GD_Score": pc2_match["GD_Score2"], "GD_BTLProxy2": 1}

    return {"GD_Index": 0, "GD_Score": 0, "GD_BTLProxy2": 0}


def build_pc1_reference(experian_extract: pd.DataFrame) -> dict:
    """
    Equivalent of CTE_PC1: builds a {postcode: {GD_Index1, GD_Score1}}
    lookup from the whole Experian extract, keeping the best-ranked
    (lowest index, then lowest score) record per normalised postcode.
    `experian_extract` must already be reduced to one row per
    (Period, LenderCode, PrimaryAccount) -- i.e. the CustRank=1 filter
    from the SQL source -- before calling this.
    """
    candidates = []
    for _, row in experian_extract.iterrows():
        pc1 = normalise_postcode_pc1(row.get("PORTFOLIO_POSTCODE"))
        if is_valid_pc1_candidate(pc1):
            candidates.append({"PC1": pc1, "GD_Index1": row.get("ND_G_01"), "GD_Score1": row.get("ND_G_02")})

    if not candidates:
        return {}

    frame = pd.DataFrame(candidates)
    frame = frame.sort_values(["PC1", "GD_Index1", "GD_Score1"], ascending=[True, True, True], kind="stable")
    best_per_postcode = frame.drop_duplicates(subset="PC1", keep="first")
    return {
        row["PC1"]: {"GD_Index1": row["GD_Index1"], "GD_Score1": row["GD_Score1"]}
        for _, row in best_per_postcode.iterrows()
    }


def build_pc2_reference(pc1_lookup: dict) -> dict:
    """Equivalent of CTE_PC2: derives a district-level lookup from the PC1 lookup, same best-per-key rule."""
    candidates = [
        {"PC2": postcode_pc1_to_pc2(pc1), "GD_Index2": v["GD_Index1"], "GD_Score2": v["GD_Score1"]}
        for pc1, v in pc1_lookup.items()
    ]
    if not candidates:
        return {}
    frame = pd.DataFrame(candidates)
    frame = frame.sort_values(["PC2", "GD_Index2", "GD_Score2"], ascending=[True, True, True], kind="stable")
    best_per_district = frame.drop_duplicates(subset="PC2", keep="first")
    return {
        row["PC2"]: {"GD_Index2": row["GD_Index2"], "GD_Score2": row["GD_Score2"]}
        for _, row in best_per_district.iterrows()
    }


### 7. SQL query templates (raw bureau attribute extraction — pass-through, unchanged predicates)

In [ ]:
def _in_clause(n: int) -> str:
    if n == 0:
        return "(SELECT NULL WHERE 1 = 0)"
    return "(" + ", ".join(["?"] * n) + ")"


BUREAU_RAW_COLUMNS = [
    "SP_F3_35", "SP_F1_30", "SP_F3_36",
    "CLU_NPR_L1M", "PTBR_L6M_NPR_L6M", "PTBR_L3M_NPR_L3M", "NO_CA_L3M", "NO_MLV_CA_L3M",
    "VM01_SP_VM2_04", "VM01_SP_VM2_05", "VM01_SP_VM2_23", "VM01_SP_VM2_29",
    "VM02_SP_VM1_04", "VM02_SP_VM1_05", "VM02_SP_VM1_21",
    "VM03_SP_VM2_04", "VM03_SP_VM2_05", "VM03_SP_VM2_23", "VM03_SP_VM2_29",
    "VM04_SP_VM1_04", "VM04_SP_VM1_05", "VM04_SP_VM1_21",
    "VM05_SP_VM1_04", "VM05_SP_VM1_05", "VM05_SP_VM1_21",
    "VM06_SP_VM1_04", "VM06_SP_VM1_05", "VM06_SP_VM1_21",
    "VM07_SP_VM1_04", "VM07_SP_VM1_05", "VM07_SP_VM1_21",
    "VM08_SP_VM2_04", "VM08_SP_VM2_05", "VM08_SP_VM2_23", "VM08_SP_VM2_29",
    "VM09_SP_VM1_04", "VM09_SP_VM1_05", "VM09_SP_VM1_21",
    "VM10_SP_VM2_04", "VM10_SP_VM2_05", "VM10_SP_VM2_23", "VM10_SP_VM2_29",
    "HC_A_08", "HC_C_05", "HC_E_02",
    "PDL_C_01", "PDL_E_01", "PDL_F_01",
    "MAIN_IVA", "MAIN_BANKRUPT", "MAIN_CCJ", "EA1_D_03",
    "NPD_BAL_NPD_SP_L12M", "NPD_BAL_EBAD_SP_L12M", "NPD_BAL_NPD_SP_L18M", "NPD_BAL_EBAD_SP_L18M",
    "SP_C_24", "SP_CONSUMER_INDEBT", "E5_S_05_1",
    "PORTFOLIO_TITLE", "PORTFOLIO_GIVEN_NM", "PORTFOLIO_FAM_NM",
]


def build_applicant_bureau_query(cfg: BureauQualifierConfig, cust_rank: int, include_geo_and_lending: bool) -> str:
    """
    ONE parameterised template replacing the SAS source's two ~150-line
    near-duplicate CTE_C1 / CTE_C2 blocks (lines 369-745). `cust_rank`
    selects applicant 1 or 2 (matches the source's CustRank filter);
    `include_geo_and_lending` includes Lending/PostCode (only fetched
    once, for applicant 1, matching the original -- Geo-Delphi and
    Lending are account-level, not applicant-level attributes).
    Population predicate is byte-for-byte identical to Steps 1/2/4's
    CTE_C1/CTE_C2 WHERE clause.
    """
    company_in = _in_clause(len(cfg.company_scope))
    product_ex = _in_clause(len(cfg.excluded_product_codes))
    bureau_cols = ",\n        ex.".join(BUREAU_RAW_COLUMNS)
    geo_lending_select = (
        ", CASE WHEN dsp.SubPopulationCode = 'BTL' THEN 'BTL' ELSE 'Resi' END AS Lending"
        ", dms.PostCode"
        ", ex.ND_G_01, ex.ND_G_02"
        if include_geo_and_lending else ""
    )
    geo_lending_join = "INNER JOIN P_DW.dbo.dimSubPopulation dsp ON fma.FK_SubPopulation = dsp.PK_SubPopulation\n" \
                        "INNER JOIN P_DW.dbo.dimMortgageSecurity dms ON fma.FK_MortgageSecurity = dms.PK_MortgageSecurity\n" \
        if include_geo_and_lending else ""

    return f"""
SELECT  CASE WHEN lu.Allocated_AccountNumber IS NOT NULL THEN lu.Allocated_AccountNumber ELSE da.AccountNumber END AS AccountNumber
        {geo_lending_select}
        , ex.{bureau_cols}
FROM    P_DW.dbo.factMortgageAccountMonthSS fma
INNER JOIN  P_DW.dbo.dimAccount da ON fma.FK_Account = da.PK_Account
INNER JOIN  P_DW.dbo.dimMonth dm ON fma.FK_SnapshotDate = dm.PK_Month
INNER JOIN  P_DW.dbo.dimAccountStates das ON fma.FK_AccountStates = das.PK_AccountStates
INNER JOIN  P_DW.dbo.dimProduct dp ON fma.FK_CurrentMortgageProduct = dp.PK_Product
INNER JOIN  P_DW.dbo.dimCompany dc ON fma.FK_Company = dc.PK_Company
{geo_lending_join}LEFT JOIN   P_ODS.reference.AHLNYMMigratedAccounts lu
    ON  da.AccountPool = lu.AHLNYM_PoolID AND da.AccountNumber = lu.AHLNYM_AccountNumber
LEFT JOIN   (
                SELECT b.* FROM (
                    SELECT a.*, RANK() OVER (PARTITION BY a.Period, a.LenderCode, a.PrimaryAccount ORDER BY a.CustomerID ASC) AS CustRank
                    FROM P_ODS.EXPERIAN.DCMGen10 a
                    WHERE a.Period = ?
                ) b WHERE b.CustRank = ?
            ) ex
    ON  (CASE WHEN dc.CompanyName = 'Amber Homeloans Ltd' THEN 'AHL'
              WHEN dc.CompanyName = 'North Yorkshire Mortgages' THEN 'NYM'
              ELSE 'SBS' END) = ex.LenderCode
    AND da.AccountNumber = ex.PrimaryAccount
WHERE   dc.CompanyName IN {company_in}
AND     da.AccountPool <> ?
AND     dm.CalendarMonthYear = CONVERT(VARCHAR, ?)
AND     da.AccountSubType = 'Primary'
AND     da.CurrentAccountClassification = 'Residential'
AND     das.AccountStatus = 'Live'
AND     dp.ProductCode NOT IN {product_ex}
AND     fma.CapitalBalanceLive > ?
AND     ((dc.CompanyName IN ('Skipton Building Society') AND fma.ContractualArrearsInMonths < 1)
         OR (dc.CompanyName IN ('Amber Homeloans Ltd', 'North Yorkshire Mortgages') AND fma.StandardArrearsInMonths < 1))
AND     da.AccountStartDate <= DATEADD(MONTH, DATEDIFF(MONTH, -1, dm.BatchDate) - 1, -1)
"""


def build_applicant_bureau_params(cfg: BureauQualifierConfig, yyyymm: int, cust_rank: int) -> list:
    return (
        [yyyymm, cust_rank]
        + list(cfg.company_scope)
        + [cfg.excluded_account_pool, yyyymm]
        + list(cfg.excluded_product_codes)
        + [cfg.capital_balance_threshold]
    )


EXPERIAN_REFERENCE_QUERY = """
SELECT b.PORTFOLIO_POSTCODE, b.ND_G_01, b.ND_G_02
FROM (
    SELECT a.*, RANK() OVER (PARTITION BY a.Period, a.LenderCode, a.PrimaryAccount ORDER BY a.CustomerID ASC) AS CustRank
    FROM P_ODS.EXPERIAN.DCMGen10 a
    WHERE a.Period = ?
) b
WHERE b.CustRank = 1
"""


### 8. Orchestration (extraction + scoring + validation — NO threshold filter applied)

In [ ]:
def retry(times: int, backoff_seconds: float):
    import functools
    import time

    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            last_exc = None
            for attempt in range(1, times + 1):
                try:
                    return fn(*args, **kwargs)
                except Exception as exc:  # noqa: BLE001
                    last_exc = exc
                    logger.warning("Attempt %s/%s failed for %s: %s", attempt, times, fn.__name__, exc)
                    if attempt < times:
                        time.sleep(backoff_seconds * attempt)
            raise BureauQualifierError(f"{fn.__name__} failed after {times} attempts") from last_exc
        return wrapper
    return decorator


class DataWarehouseConnector:
    def __init__(self, cfg: BureauQualifierConfig):
        self.cfg = cfg
        self._conn = None

    def _connection_string(self) -> str:
        return (
            f"DRIVER={{{self.cfg.db_driver}}};SERVER={self.cfg.db_server};"
            f"DATABASE={self.cfg.db_database};Trusted_Connection=yes;"
            f"Connect Timeout={self.cfg.connect_timeout_seconds};"
        )

    @retry(times=3, backoff_seconds=5.0)
    def connect(self):
        import pyodbc
        logger.info("Connecting to %s/%s", self.cfg.db_server, self.cfg.db_database)
        self._conn = pyodbc.connect(self._connection_string(), timeout=self.cfg.connect_timeout_seconds)
        return self._conn

    def __enter__(self):
        return self.connect()

    def __exit__(self, exc_type, exc_val, exc_tb):
        if self._conn is not None:
            self._conn.close()
            logger.info("Connection closed")


def apply_disabled_threshold_filter(df: pd.DataFrame, cfg: BureauQualifierConfig) -> pd.DataFrame:
    """
    Documented no-op. The SAS source's threshold WHERE clause is commented
    out (see explanation §1); this function exists purely so that fact is
    visible and enforced in code, not to apply any filtering. If GBRS/SBRS/
    BGD are ever confirmed to be required here, implement the filter
    explicitly and remove this guard -- do not silently start filtering.
    """
    if any(v is not None for v in (
        cfg.gbrs_general_bureau_risk_score, cfg.sbrs_secondary_bureau_risk_score, cfg.bgd_btl_geo_delphi_threshold,
    )):
        raise BureauQualifierError(
            "GBRS/SBRS/BGD were supplied but this component does not apply the disabled "
            "threshold filter (see risk log). Confirm intent before proceeding -- do not "
            "silently start filtering a strategy population."
        )
    return df


def validate_bureau_qualifiers(df: pd.DataFrame) -> None:
    required = ["AccountNumber", "Lending", "BureauRiskScore", "ExpData", "Cust2Data", "GD_Index", "GD_BTLProxy2"]
    missing_cols = [c for c in required if c not in df.columns]
    if missing_cols:
        raise BureauQualifierError(f"Missing expected column(s): {missing_cols}")

    if df.empty:
        logger.warning("Bureau Riskier Qualifiers produced 0 rows. Confirm this is expected.")
        return

    dup_count = int(df["AccountNumber"].duplicated().sum())
    if dup_count > 0:
        raise BureauQualifierError(
            f"{dup_count} duplicate AccountNumber row(s) -- likely a RANK() tie fan-out "
            f"in the Experian CustRank lookup (see risk #4)."
        )

    bad_lending = df.loc[~df["Lending"].isin(["Resi", "BTL"])]
    if not bad_lending.empty:
        raise BureauQualifierError(f"{len(bad_lending)} row(s) with unexpected Lending value(s).")

    negative_scores = int((df["BureauRiskScore"] < 0).sum())
    if negative_scores > 0:
        raise BureauQualifierError(f"{negative_scores} row(s) with a negative BureauRiskScore -- should not be possible.")

    logger.info(
        "Validation passed: %s rows (%s Resi / %s BTL), BureauRiskScore range [%s, %s]",
        len(df), int((df["Lending"] == "Resi").sum()), int((df["Lending"] == "BTL").sum()),
        df["BureauRiskScore"].min(), df["BureauRiskScore"].max(),
    )


def extract_bureau_riskier_qualifiers(cfg: BureauQualifierConfig) -> pd.DataFrame:
    """
    Main entry point. Equivalent to Step 4's unnamed proc sql block.
    Returns one row per account with all App-level scores and
    BureauRiskScore -- UNFILTERED by risk score (see §1).
    """
    yyyymm = get_reporting_period(cfg.rundate)
    logger.info("Starting Bureau Riskier Qualifiers extraction for period %s", yyyymm)

    with DataWarehouseConnector(cfg) as conn:
        experian_ref = pd.read_sql(EXPERIAN_REFERENCE_QUERY, conn, params=[yyyymm])
        pc1_lookup = build_pc1_reference(experian_ref)
        pc2_lookup = build_pc2_reference(pc1_lookup)

        q1 = build_applicant_bureau_query(cfg, cust_rank=1, include_geo_and_lending=True)
        p1 = build_applicant_bureau_params(cfg, yyyymm, cust_rank=1)
        raw1 = pd.read_sql(q1, conn, params=p1)

        q2 = build_applicant_bureau_query(cfg, cust_rank=2, include_geo_and_lending=False)
        p2 = build_applicant_bureau_params(cfg, yyyymm, cust_rank=2)
        raw2 = pd.read_sql(q2, conn, params=p2)

    raw1_by_account = {row["AccountNumber"]: dict(row) for _, row in raw1.iterrows()}
    raw2_by_account = {row["AccountNumber"]: dict(row) for _, row in raw2.iterrows()}

    records = []
    for account_number, r1 in raw1_by_account.items():
        r1 = dict(r1)
        r1["_has_match"] = not pd.isna(r1.get("PORTFOLIO_TITLE")) if "PORTFOLIO_TITLE" in r1 else False
        r2 = dict(raw2_by_account.get(account_number, {}))
        r2["_has_match"] = bool(r2)

        applicant1_scores = score_applicant(r1)
        applicant2_scores = score_applicant(r2)
        combined = combine_applicants(applicant1_scores, applicant2_scores)

        geo = resolve_geo_delphi(
            lending=r1.get("Lending"),
            own_bureau_gd_index=r1.get("ND_G_01"),
            own_bureau_gd_score=r1.get("ND_G_02"),
            pc1_lookup=pc1_lookup,
            pc2_lookup=pc2_lookup,
            property_postcode=r1.get("PostCode"),
        )

        records.append({
            "AccountNumber": account_number,
            "Lending": r1.get("Lending"),
            **geo,
            **combined,
        })

    result = pd.DataFrame(records).sort_values("AccountNumber", kind="stable").reset_index(drop=True)
    result = apply_disabled_threshold_filter(result, cfg)  # documented no-op -- see function docstring
    validate_bureau_qualifiers(result)

    logger.info("Completed Bureau Riskier Qualifiers extraction: %s rows", len(result))
    return result


## 9. Unit tests

In [ ]:
import unittest
from datetime import date as _date


class TestConfigAndPeriod(unittest.TestCase):
    def test_rejects_non_first_of_month(self):
        with self.assertRaises(ValueError):
            BureauQualifierConfig(rundate=_date(2026, 7, 15))

    def test_reporting_period(self):
        self.assertEqual(get_reporting_period(_date(2026, 7, 1)), 202607)


class TestPostcodeNormalisation(unittest.TestCase):
    def test_seven_char_postcode(self):
        self.assertEqual(normalise_postcode_pc1("AB123CD"), "AB12 3CD")

    def test_six_char_postcode(self):
        self.assertEqual(normalise_postcode_pc1("AB12CD"), "AB1 2CD")

    def test_five_char_postcode(self):
        self.assertEqual(normalise_postcode_pc1("A12CD"), "A1 2CD")

    def test_four_char_postcode(self):
        self.assertEqual(normalise_postcode_pc1("A1CD"), "A 1CD")

    def test_already_spaced_input_is_stripped_and_reinserted(self):
        self.assertEqual(normalise_postcode_pc1("AB1 2 3CD"), "AB12 3CD")

    def test_out_of_range_length_unchanged(self):
        # cleaned length 3 -> outside the 4-7 range handled by the SQL CASE -> ELSE branch, unchanged
        self.assertEqual(normalise_postcode_pc1("ABC"), "ABC")
        # cleaned length 8 -> also outside the 4-7 range -> unchanged
        self.assertEqual(normalise_postcode_pc1("AB1234CD"), "AB1234CD")

    def test_none_input_returns_none(self):
        self.assertIsNone(normalise_postcode_pc1(None))

    def test_pc1_to_pc2_eight_char(self):
        self.assertEqual(postcode_pc1_to_pc2("AB12 3CD"), "AB12 3C")

    def test_pc1_to_pc2_five_char(self):
        self.assertEqual(postcode_pc1_to_pc2("A 1CD"), "A 1C")

    def test_pc1_to_pc2_out_of_range_unchanged(self):
        self.assertEqual(postcode_pc1_to_pc2("ABC"), "ABC")

    def test_valid_pc1_candidate_letter_start(self):
        self.assertTrue(is_valid_pc1_candidate("AB12 3CD"))

    def test_valid_pc1_candidate_digit_start_rejected(self):
        self.assertFalse(is_valid_pc1_candidate("1B12 3CD"))

    def test_valid_pc1_candidate_none_rejected(self):
        self.assertFalse(is_valid_pc1_candidate(None))

    def test_valid_pc1_candidate_empty_string_rejected(self):
        self.assertFalse(is_valid_pc1_candidate(""))


class TestScoreClu(unittest.TestCase):
    def test_all_three_conditions_scores_3(self):
        self.assertEqual(score_clu(sp_f3_35=90, sp_f1_30=3, sp_f3_36=90), 3)

    def test_first_two_conditions_only_scores_2(self):
        self.assertEqual(score_clu(sp_f3_35=90, sp_f1_30=3, sp_f3_36=50), 2)

    def test_lower_band_scores_1(self):
        self.assertEqual(score_clu(sp_f3_35=85, sp_f1_30=2, sp_f3_36=None), 1)

    def test_below_all_thresholds_scores_0(self):
        self.assertEqual(score_clu(sp_f3_35=50, sp_f1_30=1, sp_f3_36=None), 0)

    def test_upper_sentinel_9999_excluded(self):
        # SQL: SP_F3_35 < 9999 excludes the sentinel value itself
        self.assertEqual(score_clu(sp_f3_35=9999, sp_f1_30=5, sp_f3_36=9999), 0)

    def test_null_inputs_score_0(self):
        self.assertEqual(score_clu(sp_f3_35=None, sp_f1_30=None, sp_f3_36=None), 0)


class TestScoreVmRevolving(unittest.TestCase):
    def test_recent_default_within_12m_scores_3(self):
        self.assertEqual(score_vm_revolving("0", "0", 0, 6), 3)

    def test_default_13_to_24m_scores_2(self):
        self.assertEqual(score_vm_revolving("0", "0", 0, 18), 2)

    def test_bad_current_status_alone_scores_2(self):
        self.assertEqual(score_vm_revolving("0", "5", 0, None), 2)

    def test_bad_l6m_plus_high_clu_scores_1(self):
        self.assertEqual(score_vm_revolving("5", "1", 80, None), 1)

    def test_bad_l6m_but_low_clu_scores_0(self):
        self.assertEqual(score_vm_revolving("5", "1", 50, None), 0)

    def test_clean_record_scores_0(self):
        self.assertEqual(score_vm_revolving("0", "0", 0, None), 0)

    def test_all_null_scores_0(self):
        self.assertEqual(score_vm_revolving(None, None, None, None), 0)

    def test_default_at_25m_boundary_scores_0(self):
        self.assertEqual(score_vm_revolving("0", "0", 0, 25), 0)

    def test_default_at_24m_boundary_scores_2(self):
        self.assertEqual(score_vm_revolving("0", "0", 0, 24), 2)


class TestScoreVmInstalment(unittest.TestCase):
    def test_matches_revolving_pattern_minus_clu_gate(self):
        # instalment pattern's tier-1 branch has no CLU condition
        self.assertEqual(score_vm_instalment("5", "1", None), 1)

    def test_recent_default_scores_3(self):
        self.assertEqual(score_vm_instalment("0", "0", 3), 3)

    def test_clean_record_scores_0(self):
        self.assertEqual(score_vm_instalment("0", "0", None), 0)


class TestAllTenVerticalMarketsUseCorrectPattern(unittest.TestCase):
    """Confirms the DRY consolidation didn't change which pattern applies to which market."""

    def test_revolving_markets_get_clu_gated_tier1(self):
        # bad L6M + low current status + high CLU -> revolving tier1=1; without CLU gate would still be
        # 1 under instalment pattern too, so use a case that DIFFERS: high CLU required for revolving tier1
        self.assertEqual(score_vm_revolving("5", "1", 80, None), 1)  # revolving: qualifies via CLU
        self.assertEqual(score_vm_revolving("5", "1", 10, None), 0)  # revolving: fails without CLU>=75
        self.assertEqual(score_vm_instalment("5", "1", None), 1)     # instalment: no CLU gate needed


class TestScoreHc(unittest.TestCase):
    def test_recent_default_scores_3(self):
        self.assertEqual(score_hc(hc_a_08=0, hc_c_05=6, hc_e_02=0), 3)

    def test_positive_balance_scores_2(self):
        self.assertEqual(score_hc(hc_a_08=100, hc_c_05=None, hc_e_02=0), 2)

    def test_searches_only_scores_1(self):
        self.assertEqual(score_hc(hc_a_08=0, hc_c_05=None, hc_e_02=2), 1)

    def test_clean_scores_0(self):
        self.assertEqual(score_hc(hc_a_08=0, hc_c_05=None, hc_e_02=0), 0)


class TestScorePdl(unittest.TestCase):
    def test_default_scores_3(self):
        self.assertEqual(score_pdl(pdl_e_01=1, pdl_c_01=0, pdl_f_01=0), 3)

    def test_active_pdl_scores_2(self):
        self.assertEqual(score_pdl(pdl_e_01=0, pdl_c_01=1, pdl_f_01=0), 2)

    def test_searches_only_scores_1(self):
        self.assertEqual(score_pdl(pdl_e_01=0, pdl_c_01=0, pdl_f_01=1), 1)

    def test_clean_scores_0(self):
        self.assertEqual(score_pdl(pdl_e_01=0, pdl_c_01=0, pdl_f_01=0), 0)


class TestScoreBankIvaPir(unittest.TestCase):
    def test_bankrupt_scores_3(self):
        self.assertEqual(score_bank_iva_pir("B", None, 0, None), 3)

    def test_iva_scores_3(self):
        self.assertEqual(score_bank_iva_pir(None, "V", 0, None), 3)

    def test_ccj_recent_scores_2(self):
        self.assertEqual(score_bank_iva_pir("X", "X", 600, 6), 2)

    def test_ccj_old_scores_1(self):
        self.assertEqual(score_bank_iva_pir("X", "X", 600, 18), 1)

    def test_clean_scores_0(self):
        self.assertEqual(score_bank_iva_pir("X", "X", 0, None), 0)


class TestScoreNpdEb(unittest.TestCase):
    def test_npd_over_500_scores_3(self):
        self.assertEqual(score_npd_eb(600, 0, 0, 0), 3)

    def test_ebad_only_scores_2(self):
        self.assertEqual(score_npd_eb(200, 600, 0, 0), 2)

    def test_l18m_fallback_scores_1(self):
        self.assertEqual(score_npd_eb(0, 0, 600, 0), 1)

    def test_all_zero_scores_0(self):
        self.assertEqual(score_npd_eb(0, 0, 0, 0), 0)


class TestScoreCifas(unittest.TestCase):
    def test_dirty_code_scores_3(self):
        self.assertEqual(score_cifas("01"), 3)

    def test_clean_code_scores_0(self):
        self.assertEqual(score_cifas("N"), 0)

    def test_none_scores_0_per_sql_null_not_in_semantics(self):
        # SQL `NULL NOT IN (...)` is UNKNOWN, not TRUE -- reproduced, not "fixed"
        self.assertEqual(score_cifas(None), 0)


class TestCombineApplicants(unittest.TestCase):
    def _scored(self, has_match, **overrides):
        base = {name: 0 for name in SCORE_FIELD_NAMES}
        base.update(overrides)
        base["CII"] = 0
        base["AAM"] = 0
        base["has_bureau_match"] = has_match
        return base

    def test_takes_max_per_criterion(self):
        a1 = self._scored(True, CLU_Score=2, HC_Score=1)
        a2 = self._scored(True, CLU_Score=1, HC_Score=3)
        result = combine_applicants(a1, a2)
        self.assertEqual(result["CLU_Score_App"], 2)
        self.assertEqual(result["HC_Score_App"], 3)

    def test_bureau_risk_score_is_sum_of_app_scores(self):
        a1 = self._scored(True, CLU_Score=1, HC_Score=1)
        a2 = self._scored(True, CLU_Score=1, HC_Score=1)
        result = combine_applicants(a1, a2)
        self.assertEqual(result["BureauRiskScore"], 2)  # only CLU+HC nonzero, all others 0

    def test_expdata_flag_requires_applicant1_match(self):
        a1 = self._scored(False)
        a2 = self._scored(True)
        result = combine_applicants(a1, a2)
        self.assertEqual(result["ExpData"], 0)

    def test_cust2data_requires_both_applicants_matched(self):
        a1 = self._scored(True)
        a2 = self._scored(False)
        result = combine_applicants(a1, a2)
        self.assertEqual(result["Cust2Data"], 0)

        a2_matched = self._scored(True)
        result2 = combine_applicants(a1, a2_matched)
        self.assertEqual(result2["Cust2Data"], 1)

    def test_tie_defaults_to_either_value_since_equal(self):
        a1 = self._scored(True, CLU_Score=2)
        a2 = self._scored(True, CLU_Score=2)
        result = combine_applicants(a1, a2)
        self.assertEqual(result["CLU_Score_App"], 2)


class TestResolveGeoDelphi(unittest.TestCase):
    def test_resi_uses_own_bureau_record(self):
        result = resolve_geo_delphi("Resi", 5, 700, {}, {}, "AB12 3CD")
        self.assertEqual(result, {"GD_Index": 5, "GD_Score": 700, "GD_BTLProxy2": 0})

    def test_resi_missing_own_record_defaults_zero(self):
        result = resolve_geo_delphi("Resi", None, None, {}, {}, "AB12 3CD")
        self.assertEqual(result, {"GD_Index": 0, "GD_Score": 0, "GD_BTLProxy2": 0})

    def test_btl_pc1_match_no_proxy_flag(self):
        pc1_lookup = {"AB12 3CD": {"GD_Index1": 4, "GD_Score1": 500}}
        result = resolve_geo_delphi("BTL", None, None, pc1_lookup, {}, "AB12 3CD")
        self.assertEqual(result, {"GD_Index": 4, "GD_Score": 500, "GD_BTLProxy2": 0})

    def test_btl_falls_back_to_pc2_with_proxy_flag(self):
        pc2_lookup = {"AB12 3C": {"GD_Index2": 6, "GD_Score2": 800}}
        result = resolve_geo_delphi("BTL", None, None, {}, pc2_lookup, "AB12 3CD")
        self.assertEqual(result, {"GD_Index": 6, "GD_Score": 800, "GD_BTLProxy2": 1})

    def test_btl_no_match_anywhere_defaults_zero(self):
        result = resolve_geo_delphi("BTL", None, None, {}, {}, "AB12 3CD")
        self.assertEqual(result, {"GD_Index": 0, "GD_Score": 0, "GD_BTLProxy2": 0})


class TestPc1Pc2ReferenceBuilders(unittest.TestCase):
    def test_pc1_keeps_best_ranked_row_per_postcode(self):
        extract = pd.DataFrame([
            {"PORTFOLIO_POSTCODE": "AB123CD", "ND_G_01": 5, "ND_G_02": 900},
            {"PORTFOLIO_POSTCODE": "AB123CD", "ND_G_01": 2, "ND_G_02": 100},  # lower index wins
        ])
        lookup = build_pc1_reference(extract)
        self.assertEqual(lookup["AB12 3CD"]["GD_Index1"], 2)

    def test_pc1_excludes_digit_start_postcodes(self):
        extract = pd.DataFrame([{"PORTFOLIO_POSTCODE": "1B123CD", "ND_G_01": 2, "ND_G_02": 100}])
        lookup = build_pc1_reference(extract)
        self.assertEqual(lookup, {})

    def test_pc2_derives_from_pc1(self):
        pc1_lookup = {"AB12 3CD": {"GD_Index1": 3, "GD_Score1": 400}}
        pc2_lookup = build_pc2_reference(pc1_lookup)
        self.assertIn("AB12 3C", pc2_lookup)
        self.assertEqual(pc2_lookup["AB12 3C"]["GD_Index2"], 3)


class TestQueryBuilders(unittest.TestCase):
    def setUp(self):
        self.cfg = BureauQualifierConfig(rundate=_date(2026, 7, 1))

    def test_applicant1_query_includes_geo_and_lending(self):
        query = build_applicant_bureau_query(self.cfg, cust_rank=1, include_geo_and_lending=True)
        self.assertIn("Lending", query)
        self.assertIn("PostCode", query)

    def test_applicant2_query_excludes_geo_and_lending(self):
        query = build_applicant_bureau_query(self.cfg, cust_rank=2, include_geo_and_lending=False)
        self.assertNotIn("AS Lending", query)

    def test_placeholder_count_matches_param_count(self):
        query = build_applicant_bureau_query(self.cfg, cust_rank=1, include_geo_and_lending=True)
        params = build_applicant_bureau_params(self.cfg, 202607, cust_rank=1)
        self.assertEqual(query.count("?"), len(params))


class TestDisabledThresholdFilterGuard(unittest.TestCase):
    def test_no_op_when_thresholds_unset(self):
        cfg = BureauQualifierConfig(rundate=_date(2026, 7, 1))
        df = pd.DataFrame({"AccountNumber": [1], "BureauRiskScore": [10]})
        result = apply_disabled_threshold_filter(df, cfg)
        pd.testing.assert_frame_equal(result, df)

    def test_raises_if_thresholds_are_supplied(self):
        cfg = BureauQualifierConfig(rundate=_date(2026, 7, 1), gbrs_general_bureau_risk_score=10)
        df = pd.DataFrame({"AccountNumber": [1], "BureauRiskScore": [10]})
        with self.assertRaises(BureauQualifierError):
            apply_disabled_threshold_filter(df, cfg)


class TestValidation(unittest.TestCase):
    def _valid_row(self):
        return {
            "AccountNumber": 1, "Lending": "Resi", "BureauRiskScore": 5,
            "ExpData": 1, "Cust2Data": 0, "GD_Index": 0, "GD_BTLProxy2": 0,
        }

    def test_valid_frame_passes(self):
        df = pd.DataFrame([self._valid_row()])
        validate_bureau_qualifiers(df)  # should not raise

    def test_duplicate_account_raises(self):
        df = pd.DataFrame([self._valid_row(), self._valid_row()])
        with self.assertRaises(BureauQualifierError):
            validate_bureau_qualifiers(df)

    def test_bad_lending_value_raises(self):
        row = self._valid_row()
        row["Lending"] = "Unknown"
        df = pd.DataFrame([row])
        with self.assertRaises(BureauQualifierError):
            validate_bureau_qualifiers(df)

    def test_negative_score_raises(self):
        row = self._valid_row()
        row["BureauRiskScore"] = -1
        df = pd.DataFrame([row])
        with self.assertRaises(BureauQualifierError):
            validate_bureau_qualifiers(df)

    def test_empty_frame_does_not_raise(self):
        df = pd.DataFrame(columns=["AccountNumber", "Lending", "BureauRiskScore", "ExpData", "Cust2Data", "GD_Index", "GD_BTLProxy2"])
        validate_bureau_qualifiers(df)  # should not raise, only warns


suite = unittest.TestLoader().loadTestsFromModule(sys.modules[__name__])
runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)
assert result.wasSuccessful(), "Unit tests failed — see output above."


## 10. Source-to-target reconciliation

In [ ]:
RECONCILIATION_QUERIES = {
    "control_totals": """
        SELECT 'legacy_sas' AS source, COUNT(*) AS row_count,
               SUM(BureauRiskScore) AS sum_score, AVG(CAST(BureauRiskScore AS FLOAT)) AS avg_score,
               SUM(CASE WHEN Lending='BTL' THEN 1 ELSE 0 END) AS btl_count
        FROM staging.BR_{yyyymm}_legacy
        UNION ALL
        SELECT 'python' AS source, COUNT(*) AS row_count,
               SUM(BureauRiskScore) AS sum_score, AVG(CAST(BureauRiskScore AS FLOAT)) AS avg_score,
               SUM(CASE WHEN Lending='BTL' THEN 1 ELSE 0 END) AS btl_count
        FROM staging.BR_{yyyymm}_python;
    """,
    "accounts_in_legacy_not_python": """
        SELECT l.AccountNumber
        FROM staging.BR_{yyyymm}_legacy l
        LEFT JOIN staging.BR_{yyyymm}_python p ON l.AccountNumber = p.AccountNumber
        WHERE p.AccountNumber IS NULL;
    """,
    "score_level_mismatches": """
        SELECT l.AccountNumber,
               l.BureauRiskScore AS legacy_score, p.BureauRiskScore AS python_score,
               l.GD_Index AS legacy_gd_index, p.GD_Index AS python_gd_index,
               l.ExpData AS legacy_expdata, p.ExpData AS python_expdata
        FROM staging.BR_{yyyymm}_legacy l
        INNER JOIN staging.BR_{yyyymm}_python p ON l.AccountNumber = p.AccountNumber
        WHERE l.BureauRiskScore <> p.BureauRiskScore
           OR l.GD_Index        <> p.GD_Index
           OR l.ExpData         <> p.ExpData;
    """,
}


def render_reconciliation_queries(yyyymm: int) -> dict:
    return {name: sql.format(yyyymm=yyyymm) for name, sql in RECONCILIATION_QUERIES.items()}


def reconcile_dataframes(legacy_df: pd.DataFrame, python_df: pd.DataFrame) -> dict:
    compare_cols = ["BureauRiskScore", "GD_Index", "GD_BTLProxy2", "ExpData", "Cust2Data"]
    merged = legacy_df.merge(python_df, on="AccountNumber", how="outer", suffixes=("_legacy", "_python"), indicator=True)

    mismatch_mask = pd.Series(False, index=merged.index)
    for col in compare_cols:
        if f"{col}_legacy" in merged.columns:
            mismatch_mask |= merged[f"{col}_legacy"] != merged[f"{col}_python"]

    return {
        "control_totals": pd.DataFrame([
            {"source": "legacy", "row_count": len(legacy_df), "sum_score": legacy_df["BureauRiskScore"].sum()},
            {"source": "python", "row_count": len(python_df), "sum_score": python_df["BureauRiskScore"].sum()},
        ]),
        "accounts_in_legacy_not_python": merged.loc[merged["_merge"] == "left_only", ["AccountNumber"]],
        "accounts_in_python_not_legacy": merged.loc[merged["_merge"] == "right_only", ["AccountNumber"]],
        "score_level_mismatches": merged.loc[(merged["_merge"] == "both") & mismatch_mask],
    }


## 11. Consolidated `TODO: VERIFY` list

- **Whether/where the `GBRS`/`SBRS`/`BGD` threshold cut is actually
  applied** — the SAS source's own WHERE clause referencing them is
  commented out; confirm this happens in Step 6 ("merge/cut-offs") or
  is genuinely no longer applied anywhere.
- **Format of `dimMortgageSecurity.PostCode`** — confirm it matches the
  Experian-normalised "OUTWARD INWARD" single-space format used for the
  `CTE_C1` → `p1.PC1` join, or BTL Geo-Delphi proxying may be silently
  under-matching today (pre-existing, not introduced by this
  conversion).
- **Uniqueness of `CustomerID` per (Period, LenderCode, PrimaryAccount)**
  in `P_ODS.EXPERIAN.DCMGen10` — a tie would fan out both the applicant
  1/2 selection and the postcode reference table.
- **`ODBC Driver 17 for SQL Server`** — as with prior steps, confirm the
  exact driver available in production.
